In [1]:
import scripts.stock_plots as stock_plots
import scripts.stock_analysis as stock_ana
import plotly.graph_objects as go

from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import yfinance as yf

import requests
from bs4 import BeautifulSoup

pd.set_option("mode.chained_assignment", None)  # Ignore the warning

In [2]:
url = "https://www.tradingview.com/markets/stocks-usa/market-movers-active/"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")
table = soup.find("table")


In [3]:
data = pd.DataFrame(columns=["Ticker", "Vol*Price (M)", "Market cap (M)", "change"])


if response.status_code == 200:
    # Parse the HTML content of the webpage
    soup = BeautifulSoup(response.content, "html.parser")

    # Find the table element on the webpage (you need to inspect the webpage to find the specific HTML tags)
    table = soup.find("table")  # Example: finding the first table on the page

    # Check if the table element was found
    if table:
        # Extract the table rows
        rows = table.find_all("tr")

        # Loop through each row and extract the data
        for row in rows:

            ticker_cell = row.find("td", class_="cell-RLhfr_y4")
            if ticker_cell:
              ticker_symbol = ticker_cell.find("a", class_="tickerName-GrtoTeat").text.strip()
              # print(ticker_symbol)


              value_1_cell = row.find_all("td")[1]
              chg_cell = row.find_all("td")[3]
              value_2_cell = row.find_all("td")[6]

              value_1 = value_1_cell.text.strip().split()
              chg_val = chg_cell.text.strip()
              value_2 = value_2_cell.text.strip().split()


              if len(value_1) == len(value_2):
                def to_million(extract):
                  if extract[1] == "T":
                    return float(extract[0]) * 1e6
                  elif extract[1] == "B":
                    return float(extract[0]) * 1e3
                  else:
                    return float(extract[0])

                vol_price = to_million(value_1)
                market_cap = to_million(value_2)


                if chg_val[0] == "+":
                  chg = float(chg_val[1:-1])
                else:
                  chg = -float(chg_val[1:-1])

                if market_cap < 2000:
                  pass
                else:
                  # print(vol_price, market_cap, relative_active)
                  data.loc[len(data.index)] = [ticker_symbol, vol_price, market_cap, chg]


                  # data = data.append({"Ticker": ticker_symbol, "Vol*Price (M)": vol_price, "Market cap (M)": market_cap, "relative_active": relative_active}, ignore_index=True)
                  # data = pd.concat([ticker_symbol, vol_price, market_cap, relative_active], ignore_index=True)

    else:
        print("Table element not found on the webpage.")
else:
    print("Failed to retrieve webpage. Status code:", response.status_code)

In [4]:
data["rel_active"] = data["Vol*Price (M)"] / data["Market cap (M)"] * (abs(data["change"]))
data["rel_active(sqrt)"] = data["Vol*Price (M)"] / np.sqrt(data["Market cap (M)"]) * (abs(data["change"]))
data["rel_active(log)"] = data["Vol*Price (M)"] / np.log(data["Market cap (M)"]) * (abs(data["change"]))


data["rel_active() rank"] = data["rel_active"].rank(ascending=False)
data["rel_active(sqrt) rank"] = data["rel_active(sqrt)"].rank(ascending=False)
data["rel_active(log) rank"] = data["rel_active(log)"].rank(ascending=False)

data["rel_rank"] = data["rel_active() rank"]**2 + data["rel_active(sqrt) rank"]**2 + data["rel_active(log) rank"]**2
data["rel_active(rel_rank) rank"] = data["rel_rank"].rank()


In [5]:
data.sort_values(by="rel_active(rel_rank) rank").head(10)

,Ticker,Vol*Price (M),Market cap (M),change,rel_active,rel_active(sqrt),rel_active(log),rel_active() rank,rel_active(sqrt) rank,rel_active(log) rank,rel_rank,rel_active(rel_rank) rank
22,INSM,1606.0,3269.0,118.45,58.192322,3327.155859,23507.794374,1.0,1.0,2.0,6.0,1.0
14,GME,2499.0,7281.0,25.16,8.635468,736.853885,7070.130877,2.0,2.0,3.0,17.0,2.0
29,CELH,1450.0,19327.0,-12.85,0.964066,134.025954,1887.933147,3.0,4.0,5.0,50.0,3.0
0,NVDA,74343.0,2848000.0,6.98,0.182203,307.486117,34915.198931,7.0,3.0,1.0,59.0,4.0
26,DKNG,1545.0,17435.0,-10.29,0.911847,120.401773,1627.858644,4.0,5.0,6.0,77.0,5.0
34,HUBS,1261.0,32534.0,8.17,0.316665,57.117398,991.562018,6.0,7.0,8.0,149.0,6.0
1,AMD,11411.0,277376.0,3.16,0.130000,68.466219,2877.075568,10.0,6.0,4.0,152.0,7.0
50,MRNA,1017.0,58712.0,-8.05,0.139441,33.787293,745.587632,8.0,8.0,11.0,249.0,8.0
12,COIN,2656.0,60164.0,3.09,0.136411,33.459404,745.767123,9.0,9.0,10.0,262.0,9.0
9,DELL,3104.0,118546.0,3.68,0.096357,33.176157,977.716760,14.0,10.0,9.0,377.0,10.0


In [6]:
def download_data(stocks):
    stocks_str = " ".join(stocks)
    return yf.Tickers(stocks_str)

In [7]:
selected_tickers = data.sort_values(by="rel_active(rel_rank) rank")["Ticker"].head(50).values

stock_data = download_data(selected_tickers)



for ticker in selected_tickers:
    stock_plot = stock_plots.PlotInfo(stock_data.tickers[ticker], ticker, "150d")
    try:
        candle = stock_plot.generate_recent_candles(p2p_order=4, days=50)
        # candle = stock_plot.generate_candle_plot(p2p_order=4)
        # candle.show()
    except:
        pass